#Transform Races Data
 1. Read Bronze races Data
 2. Keep only columns required for analytics ( Drop url column)
 3. Standardise column names using snake_case(circuitId -> circuit_Id, raceName -> race_name)
 4. Rename Columns to make them more meaningful (date->race_date)
 6. Remove duplicate records
 7. Transform values of columns race_name and  to Title case
 8. Write the transformed data to silver races table


In [0]:
%run ../00_common/01_environment_config

In [0]:
%run ../00_common/01_environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_read_df = spark.table(bronze_table)
display(races_read_df)

In [0]:
from pyspark.sql import functions as F
races_select_df = races_read_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName").alias("race_name"),
    F.col("date").alias("race_date"),
    F.col("circuitId").alias("circuit_id"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
    )

In [0]:
from pyspark.sql import functions as F
rm_dup_races_df = races_select_df.dropDuplicates(["season","round"])
display(rm_dup_races_df)

In [0]:
from pyspark.sql import functions as F
races_final_df = (
                    rm_dup_races_df
                    .withColumn(("race_name"),F.initcap(F.col("race_name")))
                    
)
display(races_final_df)

In [0]:
(
    races_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))